<a href="https://colab.research.google.com/github/JeetRaman21/Demo/blob/main/Day7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
from google.colab import userdata
API_KEY = userdata.get('GROQ_API_KEY')

In [ ]:
!pip install groq langchain langchain-groq langchain-community pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 60.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 76.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.9 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: groq
    Found existing installation: groq 1.5.0
    Uninstalling groq-1.5.0:
      Successfully uninstalled groq-1.5.0


In [17]:
API_KEY = userdata.get('GROQ_API_KEY')

# Quick sanity check — make sure the key loaded
assert API_KEY.startswith("gsk_"), "❌ Groq key not found — check your .env file!"
print("✅ Groq API key loaded successfully")

✅ Groq API key loaded successfully


In [19]:
from groq import Groq

client = Groq(api_key=API_KEY)

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",       # a strong, free Groq model
    messages=[
        {"role": "user", "content": "Say hello in one short sentence."}
    ],
)

print(response.choices[0].message.content)
# Expected: something like "Hello! How can I help you today?"

Hello, it's nice to meet you.


In [20]:
import math

def calculator(expression: str) -> str:
    """Safely evaluate a math expression like '4871 * 209'."""
    try:
        # We only allow math — no arbitrary code. (More on safety in Session 3.)
        allowed = {k: v for k, v in math.__dict__.items() if not k.startswith("_")}
        result = eval(expression, {"__builtins__": {}}, allowed)
        return str(result)
    except Exception as e:
        return f"Error: {e}"

def lookup_fact(query: str) -> str:
    """A pretend 'search engine' — a tiny fixed fact table for the demo."""
    facts = {
        "largest cities in punjab": "Ludhiana, Amritsar, and Jalandhar.",
        "ludhiana population": "1,618,879",
        "ludhiana area": "310 square kilometres",
    }
    key = query.lower().strip()
    return facts.get(key, "No fact found. Try a different query.")

# Quick test — always test tools before wiring them into an agent!
print(calculator("4871 * 209"))     # Expected: 1018039
print(lookup_fact("Ludhiana population"))   # Expected: 1,618,879

1018039
1,618,879


In [21]:
SYSTEM_PROMPT = """You are a helpful assistant that solves problems step by step.

You have access to these tools:
- calculator(expression): evaluates a math expression, e.g. "12 * 7"
- lookup_fact(query): looks up a fact, e.g. "Ludhiana population"

To solve a task, ALWAYS reply in this exact format:

Thought: <your reasoning about what to do next>
Action: <one of: calculator, lookup_fact>
Action Input: <the input to the tool>

After you receive an Observation, continue reasoning.
When you have the final answer, reply in EXACTLY this format instead:

Thought: I now know the final answer.
Final Answer: <your answer to the user>

Only output ONE Thought/Action/Action Input block at a time. Then stop and wait.
"""

In [22]:
pip install Groq

In [28]:
from groq import Groq

client = Groq(api_key=API_KEY)

def ask_llm(messages):
    """Send the conversation so far and get the model's next step."""
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=messages,
        temperature=0,           # 0 = deterministic; we want reliable, not creative
        stop=["Observation:"],   # stop the model BEFORE it hallucinates its own observation
    )
    return response.choices[0].message.content

In [30]:
import re

def run_agent(user_question, max_iterations=6):
    # The running transcript. Starts with the rules + the user's question.
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_question},
    ]

    tools = {"calculator": calculator, "lookup_fact": lookup_fact}

    for step in range(max_iterations):        # max_iterations = a safety cap (Session 3!)
        # 1) Ask the model for its next Thought + Action
        reply = ask_llm(messages)
        print(f"\n--- Iteration {step + 1} ---")
        print(reply)

        # 2) Did the model give a Final Answer? If so, we're done.
        if "Final Answer:" in reply:
            answer = reply.split("Final Answer:")[-1].strip()
            return answer

        # 3) Otherwise, extract which tool it wants and the input
        action_match = re.search(r"Action:\s*(.+)", reply)
        input_match = re.search(r"Action Input:\s*(.+)", reply)
        if not action_match or not input_match:
            return "⚠️ Model did not follow the format. Stopping."

        tool_name = action_match.group(1).strip()
        tool_input = input_match.group(1).strip()

        # 4) Run the real tool
        if tool_name in tools:
            observation = tools[tool_name](tool_input)
        else:
            observation = f"Error: unknown tool '{tool_name}'"

        print(f"Observation: {observation}")

        # 5) Feed the model's own reply + the real observation back in, then loop
        messages.append({"role": "assistant", "content": reply})
        messages.append({"role": "user", "content": f"Observation: {observation}"})

    return "⚠️ Ran out of iterations without a final answer."

In [31]:
question = "What is the population of Ludhiana, and what is that number multiplied by 2?"
final = run_agent(question)
print("\n========================")
print("FINAL:", final)


--- Iteration 1 ---
Thought: To find the population of Ludhiana multiplied by 2, I first need to look up the population of Ludhiana.
Action: lookup_fact
Action Input: Ludhiana population
Observation: 1,618,879

--- Iteration 2 ---
Thought: Now that I have the population of Ludhiana, I can calculate the result of multiplying this number by 2 to get the final answer.
Action: calculator
Action Input: 1618879 * 2
Observation: 3237758

--- Iteration 3 ---
Thought: I now know the final answer.
Final Answer: 3237758

FINAL: 3237758
